In [ ]:

import json
import os
import math

# File path
FILE_NAME = os.path.join(os.getcwd(), "students.json")


# ==================== FILE HANDLING ====================

def load_students():
    try:
        with open(FILE_NAME, "r", encoding="utf-8") as file:
            data = json.load(file)

        if not isinstance(data, list):
            print("Invalid JSON format. Starting with empty records.")
            return []

        students = []

        for student in data:
            if not isinstance(student, dict):
                continue

            if not all(
                key in student
                for key in ("id", "name", "age", "course", "marks")
            ):
                continue

            if not all(
                isinstance(student[key], str)
                for key in ("id", "name", "course")
            ):
                continue

            if (
                not student["id"]
                or not student["id"].isascii()
                or not student["id"].isalnum()
                or not student["name"]
                or not is_valid_name(student["name"])
                or not student["course"].strip()
                or not any(c.isalpha() for c in student["course"])
            ):
                continue

            age = student["age"]
            marks = student["marks"]

            if (
                isinstance(age, bool)
                or not isinstance(age, int)
                or not 1 <= age <= 120
            ):
                continue

            if (
                isinstance(marks, bool)
                or not isinstance(marks, (int, float))
                or not math.isfinite(marks)
                or not 0 <= marks <= 100
            ):
                continue

            # Avoid duplicate IDs in existing JSON data
            if find_student(students, student["id"]):
                continue

            students.append(student)

        return students

    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        print("Warning: students.json contains invalid JSON.")
        return []
    except OSError as error:
        print(f"Error reading records: {error}")
        return []


def save_students(students):
    try:
        with open(FILE_NAME, "w", encoding="utf-8") as file:
            json.dump(
                students,
                file,
                indent=4,
                ensure_ascii=False,
                allow_nan=False
            )
        return True

    except (OSError, ValueError) as error:
        print(f"Error saving records: {error}")
        return False


# ==================== INPUT VALIDATION ====================

def get_non_empty_text(prompt, field_name):
    while True:
        value = input(prompt).strip()

        if value:
            return value

        print(f"{field_name} cannot be empty.")


def is_valid_name(name):
    return (
        bool(name)
        and any(char.isalpha() for char in name)
        and all(
            char.isalpha() or char in " -'."
            for char in name
        )
    )


def get_student_name(prompt):
    while True:
        name = input(prompt).strip()

        if is_valid_name(name):
            return name

        print(
            "Invalid name! Use letters, spaces, hyphens, "
            "apostrophes or periods only."
        )


def get_student_id(prompt):
    while True:
        student_id = input(prompt).strip()

        if student_id and student_id.isascii() and student_id.isalnum():
            return student_id

        print(
            "Invalid Student ID! Use English letters and numbers only. "
            "Example: ST101"
        )


def get_course(prompt):
    while True:
        course = input(prompt).strip()

        if course and any(char.isalpha() for char in course):
            return course

        print("Invalid course! Enter a course containing letters.")


def get_integer(prompt, minimum, maximum):
    while True:
        value = input(prompt).strip()

        try:
            number = int(value)

            if minimum <= number <= maximum:
                return number

            print(f"Enter a whole number from {minimum} to {maximum}.")

        except ValueError:
            print("Invalid input! Enter a whole number only.")


def get_number(prompt, minimum, maximum):
    while True:
        value = input(prompt).strip()

        try:
            number = float(value)

            if not math.isfinite(number):
                print("Invalid input! Enter a finite number.")
                continue

            if minimum <= number <= maximum:
                return number

            print(f"Enter a value from {minimum} to {maximum}.")

        except ValueError:
            print("Invalid input! Enter a valid number.")


def get_yes_no(prompt):
    while True:
        answer = input(prompt).strip().lower()

        if answer in ("y", "n"):
            return answer

        print("Invalid choice! Enter y or n.")


# ==================== STUDENT SEARCH ====================

def find_student(students, student_id):
    for student in students:
        if student["id"].casefold() == student_id.casefold():
            return student

    return None


# ==================== GRADES AND RESULTS ====================

def get_grade(marks):
    if marks >= 90:
        return "A"
    elif marks >= 80:
        return "B"
    elif marks >= 70:
        return "C"
    elif marks >= 60:
        return "D"
    elif marks >= 35:
        return "E"
    else:
        return "F"


def show_result(student):
    name = student["name"]
    course = student["course"]
    marks = student["marks"]

    print("\n========== STUDENT RESULT ==========")

    if marks < 35:
        print("Result: FAIL")
        print(
            f"You have failed in {course} "
            f"with {marks:g} marks, {name}."
        )

    elif marks >= 85:
        print("Result: PASS")
        print(
            f"Congratulations, {name}! You scored {marks:g} "
            f"marks in {course}. Excellent performance!"
        )

    else:
        print("Result: PASS")
        print(
            f"Well done, {name}! You scored {marks:g} "
            f"marks in {course}."
        )

    print("Grade:", get_grade(marks))


# ==================== ADD STUDENT ====================

def add_student(students):
    student_id = get_student_id("Student ID: ")

    if find_student(students, student_id):
        print("Student ID already exists.")
        return

    name = get_student_name("Student name: ")
    age = get_integer("Age (1-120): ", 1, 120)
    course = get_course("Course: ")
    marks = get_number("Marks (0-100): ", 0, 100)

    student = {
        "id": student_id,
        "name": name,
        "age": age,
        "course": course,
        "marks": marks
    }

    students.append(student)

    if save_students(students):
        print("Student added successfully!")
        show_result(student)
    else:
        students.remove(student)
        print("Student was not added because saving failed.")


# ==================== VIEW STUDENTS ====================

def view_students(students):
    if not students:
        print("No student records found.")
        return

    print(
        f"\n{'ID':<12}{'Name':<22}{'Age':<6}"
        f"{'Course':<22}{'Marks':<10}{'Grade'}"
    )
    print("-" * 80)

    for student in students:
        print(
            f"{student['id']:<12}"
            f"{student['name']:<22}"
            f"{student['age']:<6}"
            f"{student['course']:<22}"
            f"{student['marks']:<10.1f}"
            f"{get_grade(student['marks'])}"
        )


# ==================== SEARCH STUDENT ====================

def search_student(students):
    student_id = get_student_id("Enter student ID: ")
    student = find_student(students, student_id)

    if student:
        print("\nStudent details:")
        print("ID:", student["id"])
        print("Name:", student["name"])
        print("Age:", student["age"])
        print("Course:", student["course"])
        print("Marks:", student["marks"])

        show_result(student)
    else:
        print("Student not found.")


# ==================== UPDATE STUDENT ====================

def update_student(students):
    student_id = get_student_id("Enter student ID to update: ")
    student = find_student(students, student_id)

    if not student:
        print("Student not found.")
        return

    original = student.copy()

    print("Press Enter to keep the current value.")

    name = input(f"Name [{student['name']}]: ").strip()

    if name:
        while not is_valid_name(name):
            print("Invalid name! Please use valid name characters.")
            name = input("Enter name again (or Enter to skip): ").strip()

            if not name:
                break

        if name:
            student["name"] = name

    course = input(f"Course [{student['course']}]: ").strip()

    if course:
        while not any(char.isalpha() for char in course):
            print("Invalid course! Enter a course containing letters.")
            course = input("Enter course again (or Enter to skip): ").strip()

            if not course:
                break

        if course:
            student["course"] = course

    if get_yes_no("Update age? (y/n): ") == "y":
        student["age"] = get_integer("New age (1-120): ", 1, 120)

    if get_yes_no("Update marks? (y/n): ") == "y":
        student["marks"] = get_number(
            "New marks (0-100): ", 0, 100
        )

    if save_students(students):
        print("Student updated successfully!")
        show_result(student)
    else:
        student.update(original)
        print("Update cancelled because saving failed.")


# ==================== DELETE STUDENT ====================

def delete_student(students):
    student_id = get_student_id("Enter student ID to delete: ")
    student = find_student(students, student_id)

    if not student:
        print("Student not found.")
        return

    if get_yes_no(
        f"Delete {student['name']}? (y/n): "
    ) == "y":
        index = students.index(student)
        students.pop(index)

        if save_students(students):
            print("Student deleted successfully!")
        else:
            students.insert(index, student)
            print("Deletion cancelled because saving failed.")
    else:
        print("Deletion cancelled.")


# ==================== STATISTICS ====================

def statistics(students):
    if not students:
        print("No records available.")
        return

    average = sum(s["marks"] for s in students) / len(students)
    topper = max(students, key=lambda s: s["marks"])
    failures = sum(1 for s in students if s["marks"] < 35)

    print("\n========== STUDENT STATISTICS ==========")
    print("Total students:", len(students))
    print(f"Average marks: {average:.2f}")
    print("Topper:", topper["name"])
    print("Highest marks:", topper["marks"])
    print("Topper's grade:", get_grade(topper["marks"]))
    print("Students who failed:", failures)


# ==================== SORT STUDENTS ====================

def sort_students(students):
    if not students:
        print("No student records found.")
        return

    print("1. Sort by name")
    print("2. Sort by marks (highest first)")

    choice = input("Choose (1-2): ").strip()

    if choice == "1":
        ordered = sorted(
            students,
            key=lambda s: s["name"].casefold()
        )
    elif choice == "2":
        ordered = sorted(
            students,
            key=lambda s: s["marks"],
            reverse=True
        )
    else:
        print("Invalid choice! Enter 1 or 2.")
        return

    view_students(ordered)


# ==================== AUTOMATIC TESTS ====================

def run_tests():
    print("\n========== RUNNING TESTS ==========")

    # Test ID validation rules
    valid_ids = ["ST101", "KLU2026"]
    invalid_ids = ["ST@101", "ST 101", "ST#101", ""]

    for student_id in valid_ids:
        assert (
            student_id.isascii() and student_id.isalnum()
        )
        print(f"PASS: Valid ID - {student_id}")

    for student_id in invalid_ids:
        assert not (
            student_id
            and student_id.isascii()
            and student_id.isalnum()
        )
        print(f"PASS: Invalid ID rejected - {student_id!r}")

    # Test name validation
    assert is_valid_name("Charan Sai")
    assert is_valid_name("Anne-Marie")
    assert not is_valid_name("Charan123")
    assert not is_valid_name("12345")
    print("PASS: Student name validation")

    # Test grades and result boundaries
    grade_tests = {
        0: "F",
        34: "F",
        35: "E",
        59: "E",
        60: "D",
        70: "C",
        80: "B",
        84: "B",
        85: "B",
        90: "A",
        100: "A"
    }

    for marks, expected in grade_tests.items():
        assert get_grade(marks) == expected
        print(f"PASS: {marks} marks -> Grade {expected}")

    # Test student feedback without requesting keyboard input
    import contextlib
    import io

    sample = {
        "id": "ST101",
        "name": "Ravi",
        "age": 20,
        "course": "Computer Science",
        "marks": 34
    }

    output = io.StringIO()

    with contextlib.redirect_stdout(output):
        show_result(sample)

    assert "FAIL" in output.getvalue()
    assert "Computer Science" in output.getvalue()
    assert "34" in output.getvalue()

    sample["marks"] = 85
    output = io.StringIO()

    with contextlib.redirect_stdout(output):
        show_result(sample)

    assert "Congratulations, Ravi!" in output.getvalue()
    assert "85" in output.getvalue()

    print("PASS: Fail message")
    print("PASS: Congratulations message")
    print("\nAll automatic tests passed!")


# ==================== MAIN MENU ====================

def main():
    students = load_students()

    while True:
        print("\n===== STUDENT MANAGEMENT SYSTEM =====")
        print("1. Add Student")
        print("2. View Students")
        print("3. Search Student")
        print("4. Update Student")
        print("5. Delete Student")
        print("6. Student Statistics")
        print("7. Sort Students")
        print("8. Run Automatic Tests")
        print("9. Exit")

        choice = input("Enter your choice (1-9): ").strip()

        if choice == "1":
            add_student(students)
        elif choice == "2":
            view_students(students)
        elif choice == "3":
            search_student(students)
        elif choice == "4":
            update_student(students)
        elif choice == "5":
            delete_student(students)
        elif choice == "6":
            statistics(students)
        elif choice == "7":
            sort_students(students)
        elif choice == "8":
            run_tests()
        elif choice == "9":
            if save_students(students):
                print("Records saved. Goodbye!")
            else:
                print("Warning: records could not be saved.")
            break
        else:
            print("Invalid choice! Enter a number from 1 to 9.")


if __name__ == "__main__":
    main()



===== STUDENT MANAGEMENT SYSTEM =====
1. Add Student
2. View Students
3. Search Student
4. Update Student
5. Delete Student
6. Student Statistics
7. Sort Students
8. Run Automatic Tests
9. Exit


Enter your choice (1-9):  1
Student ID:  charan
Student name:  wef
Age (1-120):  4
Course:  vfb
Marks (0-100):  5


Student added successfully!

========== STUDENT RESULT ==========
Result: FAIL
You have failed in vfb with 5 marks, wef.
Grade: F

===== STUDENT MANAGEMENT SYSTEM =====
1. Add Student
2. View Students
3. Search Student
4. Update Student
5. Delete Student
6. Student Statistics
7. Sort Students
8. Run Automatic Tests
9. Exit


Enter your choice (1-9):  7


1. Sort by name
2. Sort by marks (highest first)


Choose (1-2):  1



ID          Name                  Age   Course                Marks     Grade
--------------------------------------------------------------------------------
charan      wef                   4     vfb                   5.0       F

===== STUDENT MANAGEMENT SYSTEM =====
1. Add Student
2. View Students
3. Search Student
4. Update Student
5. Delete Student
6. Student Statistics
7. Sort Students
8. Run Automatic Tests
9. Exit


Enter your choice (1-9):  5
Enter student ID to delete:  charan
Delete wef? (y/n):  y


Student deleted successfully!

===== STUDENT MANAGEMENT SYSTEM =====
1. Add Student
2. View Students
3. Search Student
4. Update Student
5. Delete Student
6. Student Statistics
7. Sort Students
8. Run Automatic Tests
9. Exit
